# SINCA — Calidad del Aire y Meteorología

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Data-Observatory/datopia-notebooks/blob/main/medio-fisico/sinca/demo.ipynb)
[![Licencia](https://img.shields.io/badge/licencia-MIT-blue.svg)](../../LICENSE)
[![Dataset](https://img.shields.io/badge/dataset-SINCA-green.svg)]()
[![Actualización](https://img.shields.io/badge/actualización-diaria-brightgreen.svg)]()
[![Python](https://img.shields.io/badge/python-3.10%2B-blue.svg)]()

---

## Descripción

Acceso al dataset de calidad del aire y meteorología de **SINCA** (Sistema de Información Nacional
de Calidad del Aire, Ministerio del Medio Ambiente de Chile) a través del **Datopia Lakehouse**.
212 estaciones activas en las 16 regiones de Chile, cobertura desde 1971 (red completa desde 2000).

El acceso es directo vía S3 con credenciales temporales, sin descarga de archivos.

### Contenido

1. Configuración y autenticación
2. Exploración del dataset (catálogo, esquema, estaciones)
3. Mapa de estaciones — PM2.5 promedio últimos 30 días
4. Series de tiempo — PM2.5 en tres estaciones representativas
5. Series de tiempo — múltiples variables en una estación
6. Análisis exploratorio mínimo
7. Modelo lineal simple — tendencia de hidrocarburos en Quintero-Puchuncaví

### Requisitos

Cuenta en el Datopia Lakehouse · Python 3.10+ · Las dependencias se instalan automáticamente

In [ ]:
# @title Instalación de dependencias
import importlib, subprocess, sys

for paquete in ["requests", "duckdb", "pandas", "matplotlib", "plotly", "numpy"]:
    if importlib.util.find_spec(paquete) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

import json, os, getpass, pathlib
import requests, duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, HTML

EN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

def show_plotly(fig):
    # include_plotlyjs=True embeds full Plotly.js inline — no CDN, works in Colab sandbox
    display(HTML(fig.to_html(include_plotlyjs=True, full_html=False)))

print("Dependencias listas.")

In [ ]:
# @title Configuración de credenciales
print(f"Entorno: {'Google Colab' if EN_COLAB else 'local'}")

URL_API = "https://ee98cnfz7e.execute-api.us-west-2.amazonaws.com/prod"

if EN_COLAB:
    EMAIL = input("Email: ").strip()
    PASSWORD = getpass.getpass("Contraseña: ")
else:
    ruta_cfg = pathlib.Path("../../config.json")
    if not ruta_cfg.exists():
        raise FileNotFoundError(
            f"Archivo de configuración no encontrado en {ruta_cfg.resolve()}.\n"
            "Copia config.example.json → config.json y completa tus credenciales."
        )
    cfg = json.loads(ruta_cfg.read_text())
    EMAIL = cfg.get("test_user", {}).get("email") or input("Email: ").strip()
    PASSWORD = cfg.get("test_user", {}).get("password") or getpass.getpass("Contraseña: ")

print(f"API    : {URL_API}")
print(f"Usuario: {EMAIL}")

---
## 1. Autenticación y conexión a S3

In [ ]:
# Iniciar sesión y obtener credenciales temporales S3
resp_login = requests.post(
    f"{URL_API}/auth/login",
    json={"email": EMAIL, "password": PASSWORD},
    timeout=30,
)
resp_login.raise_for_status()
token = resp_login.json()["id_token"]

resp_s3 = requests.post(
    f"{URL_API}/auth/session/s3",
    headers={"Authorization": f"Bearer {token}"},
    timeout=30,
)
resp_s3.raise_for_status()
creds = resp_s3.json()

print("Sesión iniciada")
print(f"  Bucket : {creds['bucket']}")
print(f"  Región : {creds['region']}")
print(f"  Expira : {creds['expires_at']}")

In [ ]:
# Configurar DuckDB con credenciales S3
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs")
con.execute(f"""
    CREATE OR REPLACE SECRET s3 (
        TYPE          S3,
        KEY_ID        '{creds["access_key_id"]}',
        SECRET        '{creds["secret_access_key"]}',
        SESSION_TOKEN '{creds["session_token"]}',
        REGION        '{creds["region"]}'
    )
""")

BASE = f"s3://{creds['bucket']}/categoria=medio-fisico/pais=cl/fuente=sinca"
print("DuckDB conectado a S3.")

---
## 2. Exploración del dataset

In [ ]:
# Metadatos del dataset de mediciones (dataset.json en la raíz del path S3)
meta = (
    con.execute(f"SELECT * FROM read_json('{BASE}/tipo=medicion-diaria/version=v1/dataset.json')")
    .df()
    .iloc[0]
    .to_dict()
)

print(meta["description"][:600] + "...")
print()
print(f"Formato      : {meta['format']}")
print(f"Particiones  : {', '.join(meta['partition_keys'])}")

cols_list_of_dicts = [dict(col) for col in meta["columns"]]
cols = pd.DataFrame(cols_list_of_dicts)[["name", "type", "comment"]]
cols

In [ ]:
# Esquema real (año más reciente)
con.execute(f"""
    DESCRIBE SELECT * FROM read_parquet('{BASE}/tipo=medicion-diaria/version=v1/year=2026/**/*.parquet')
    LIMIT 0
""").df()[["column_name", "column_type", "null"]]

In [ ]:
# Catálogo de estaciones activas (tipo=estaciones, SCD-2 — fecha_fin IS NULL = activa)
estaciones = con.execute(f"""
    SELECT estacion_id, nombre, region, comuna, lon, lat
    FROM read_parquet('{BASE}/tipo=estaciones/version=v1/part-00000.parquet')
    WHERE fecha_fin IS NULL
""").df()

print(f"Estaciones activas: {len(estaciones)}")
print(f"Sin coordenadas (geometry nula en origen): {estaciones['lon'].isna().sum()}")
estaciones["region"].value_counts()

---
## 3. Mapa de estaciones

Nota sobre particionado: el mes en curso (año/mes actual) está particionado por día
(`year=/month=/day=`); los meses ya cerrados se consolidan a un único archivo por mes
(`year=/month=`) y se eliminan los archivos diarios. El glob recursivo `**/*.parquet` sobre
`year=YYYY/**` lee ambos formatos de forma transparente — no hace falta distinguirlos en la
consulta.

In [ ]:
# PM2.5 promedio de los últimos 30 días por estación (junio + julio 2026)
pm25_estaciones = con.execute(f"""
    SELECT estacion_id, AVG(valor) AS pm25_promedio, COUNT(*) AS n_dias
    FROM read_parquet([
        '{BASE}/tipo=medicion-diaria/version=v1/year=2026/month=06/**/*.parquet',
        '{BASE}/tipo=medicion-diaria/version=v1/year=2026/month=07/**/*.parquet'
    ])
    WHERE variable_code = 'PM25' AND periodicidad = 'diario'
    GROUP BY estacion_id
""").df()

mapa_df = pm25_estaciones.merge(estaciones, on="estacion_id").dropna(subset=["lon", "lat"])
print(f"Estaciones con dato de PM2.5 en el mapa: {len(mapa_df)} / {len(estaciones)} activas")

fig_mapa = px.scatter_mapbox(
    mapa_df,
    lat="lat", lon="lon",
    color="pm25_promedio",
    size=[8] * len(mapa_df),
    size_max=9,
    hover_name="nombre",
    hover_data={"region": True, "pm25_promedio": ":.1f", "n_dias": True, "lat": False, "lon": False},
    color_continuous_scale="YlOrRd",
    zoom=3.2,
    center={"lat": -33.5, "lon": -70.9},
    mapbox_style="open-street-map",
    title="PM2.5 promedio (µg/m³) — últimos 30 días",
)
fig_mapa.update_layout(margin=dict(l=0, r=0, t=40, b=0), height=650)
show_plotly(fig_mapa)

---
## 4. Series de tiempo — PM2.5 en estaciones representativas

Tres estaciones activas con cobertura reciente, elegidas por dispersión geográfica:
La Florida (Santiago, Región Metropolitana), Bomberos (Antofagasta, minería del norte) y
Osorno (Región de los Lagos, sur).

In [ ]:
ESTACIONES_TS = ["D12", "230", "A01"]  # La Florida, Bomberos (Antofagasta), Osorno

pm25_series = con.execute(f"""
    SELECT estacion_id, CAST(fecha AS DATE) AS fecha, valor
    FROM read_parquet([
        '{BASE}/tipo=medicion-diaria/version=v1/year=2025/**/*.parquet',
        '{BASE}/tipo=medicion-diaria/version=v1/year=2026/**/*.parquet'
    ])
    WHERE estacion_id IN ({", ".join(f"'{e}'" for e in ESTACIONES_TS)})
      AND variable_code = 'PM25' AND periodicidad = 'diario'
      AND fecha >= DATE '2025-07-01' AND fecha < DATE '2026-07-01'
    ORDER BY estacion_id, fecha
""").df()

nombres = estaciones.set_index("estacion_id")["nombre"].to_dict()
print(f"Filas cargadas: {len(pm25_series):,}")

fig_series = go.Figure()
COLORES = {"D12": "#1565C0", "230": "#E65100", "A01": "#2E7D32"}
for est_id in ESTACIONES_TS:
    sub = pm25_series[pm25_series["estacion_id"] == est_id]
    fig_series.add_trace(go.Scatter(
        x=sub["fecha"], y=sub["valor"],
        mode="lines", name=nombres[est_id],
        line=dict(color=COLORES[est_id], width=1.5),
    ))

fig_series.update_layout(
    title="PM2.5 diario (µg/m³) — jul 2025 a jun 2026",
    xaxis_title="Fecha", yaxis_title="PM2.5 (µg/m³)",
    hovermode="x unified", template="plotly_white", legend_title="Estación",
)
show_plotly(fig_series)

---
## 5. Series de tiempo — múltiples variables en una estación

Perfil de contaminantes en La Florida (Santiago): PM2.5, PM10 y O3, mismo período.

In [ ]:
VARIABLES_PERFIL = {"PM25": "PM2.5", "PM10": "PM10", "0008": "O3"}

perfil = con.execute(f"""
    SELECT CAST(fecha AS DATE) AS fecha, variable_code, valor
    FROM read_parquet([
        '{BASE}/tipo=medicion-diaria/version=v1/year=2025/**/*.parquet',
        '{BASE}/tipo=medicion-diaria/version=v1/year=2026/**/*.parquet'
    ])
    WHERE estacion_id = 'D12'
      AND variable_code IN ({", ".join(f"'{v}'" for v in VARIABLES_PERFIL)})
      AND periodicidad = 'diario'
      AND fecha >= DATE '2025-07-01' AND fecha < DATE '2026-07-01'
    ORDER BY variable_code, fecha
""").df()
perfil["variable"] = perfil["variable_code"].map(VARIABLES_PERFIL)

fig_perfil, ax = plt.subplots(figsize=(11, 4.5))
for var, color in zip(VARIABLES_PERFIL.values(), ["#C62828", "#6A1B9A", "#00838F"]):
    sub = perfil[perfil["variable"] == var]
    ax.plot(sub["fecha"], sub["valor"], label=var, color=color, linewidth=1.2)

ax.set_title("La Florida (Santiago) — PM2.5, PM10, O3 diario (µg/m³)", fontweight="bold")
ax.set_xlabel("Fecha")
ax.set_ylabel("µg/m³")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Análisis exploratorio mínimo

In [ ]:
# Estadísticas descriptivas — PM2.5 en La Florida, mismo período
pm25_d12 = perfil[perfil["variable"] == "PM2.5"]["valor"]
print("PM2.5 La Florida (jul 2025 – jun 2026):")
print(pm25_d12.describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(pm25_d12, bins=30, color="#455A64", edgecolor="white")
axes[0].set_title("Distribución PM2.5 diario")
axes[0].set_xlabel("µg/m³")
axes[0].set_ylabel("Días")

calidad = con.execute(f"""
    SELECT calidad_id, COUNT(*) AS n
    FROM read_parquet('{BASE}/tipo=medicion-diaria/version=v1/year=2026/**/*.parquet')
    WHERE variable_code = 'PM25' AND periodicidad = 'diario'
    GROUP BY calidad_id ORDER BY calidad_id
""").df()
ETIQUETAS_CALIDAD = {1: "Validado", 2: "Preliminar", 3: "No validado"}
calidad["etiqueta"] = calidad["calidad_id"].map(ETIQUETAS_CALIDAD)
axes[1].bar(calidad["etiqueta"], calidad["n"], color="#00695C")
axes[1].set_title("Filas PM2.5 2026 por calidad_id")
axes[1].set_ylabel("Filas")

plt.tight_layout()
plt.show()

---
## 7. Modelo lineal simple — tendencia de hidrocarburos en Quintero-Puchuncaví

**Estación Campiche** (Región de Valparaíso), zona industrial de Quintero-Puchuncaví. Es una de
las pocas estaciones SINCA que mide hidrocarburos no metánicos (NMHC, unidad ppm) y tiene
cobertura continua desde 2000. Ajuste de una tendencia lineal simple sobre el promedio diario —
solo para demostrar capacidad de análisis, no un modelo predictivo serio (sin estacionalidad,
sin validación cruzada).

In [ ]:
nmhc = con.execute(f"""
    SELECT CAST(fecha AS DATE) AS fecha, valor
    FROM read_parquet([
        '{BASE}/tipo=medicion-diaria/version=v1/year=2024/**/*.parquet',
        '{BASE}/tipo=medicion-diaria/version=v1/year=2025/**/*.parquet',
        '{BASE}/tipo=medicion-diaria/version=v1/year=2026/**/*.parquet'
    ])
    WHERE estacion_id = '501' AND variable_code = 'NMHC' AND periodicidad = 'diario'
    ORDER BY fecha
""").df()

print(f"Días con dato: {len(nmhc)} ({nmhc['fecha'].min()} a {nmhc['fecha'].max()})")

x = (nmhc["fecha"] - nmhc["fecha"].min()).dt.days.to_numpy()
y = nmhc["valor"].to_numpy()

pendiente, intercepto = np.polyfit(x, y, 1)
y_ajustado = pendiente * x + intercepto
ss_res = np.sum((y - y_ajustado) ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
r2 = 1 - ss_res / ss_tot

print(f"Pendiente : {pendiente:+.6f} ppm/día  ({pendiente * 365:+.4f} ppm/año)")
print(f"R²        : {r2:.4f}")

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.scatter(nmhc["fecha"], y, s=4, alpha=0.3, color="#546E7A", label="Diario")
ax.plot(nmhc["fecha"], y_ajustado, color="#D84315", linewidth=2, label="Tendencia lineal (mínimos cuadrados)")
ax.set_title("Campiche (Quintero-Puchuncaví) — NMHC diario (ppm) y tendencia", fontweight="bold")
ax.set_xlabel("Fecha")
ax.set_ylabel("NMHC (ppm)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

*Datos provistos por [Data Observatory](https://dataobservatory.net) · Fuente: SINCA, Ministerio del Medio Ambiente de Chile*